# F1 Lap Time Predictor — Does Tire Age Predict Lap Time?
**GCSRM Recruitment 2026 — Option A**

This notebook predicts lap time within one F1 race and compares a baseline model with a tire-age enhanced model. It uses a stint-based train/test split to avoid data leakage.


## Step 0 — Load the data
Upload these four CSV files to the same Colab working directory:
`lap_times.csv`, `pit_stops.csv`, `results.csv`, `races.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

lap_times = pd.read_csv("lap_times.csv")
pit_stops = pd.read_csv("pit_stops.csv")
results = pd.read_csv("results.csv")
races = pd.read_csv("races.csv")

print("lap_times:", lap_times.shape)
print("pit_stops:", pit_stops.shape)
print("results:", results.shape)
print("races:", races.shape)


## Step 1 — Select a suitable race
We use the **2019 Spanish Grand Prix**, a dry race without a red flag. The code finds its race ID from the dataset.

We select up to 8 finishers who have at least two stints, so earlier stints can be training data and the final stint can be test data.


In [ ]:
race_match = races[
    (races["year"] == 2019) &
    (races["name"].str.contains("Spanish", case=False, na=False))
]

if race_match.empty:
    raise ValueError("2019 Spanish Grand Prix was not found in races.csv.")

RACE_ID = race_match.iloc[0]["raceId"]
race_name = race_match.iloc[0]["name"]

print(f"Selected race: {race_name} (raceId={RACE_ID})")

finishers = results[
    (results["raceId"] == RACE_ID) &
    (results["statusId"] == 1)
].copy()

race_laps = lap_times[
    (lap_times["raceId"] == RACE_ID) &
    (lap_times["driverId"].isin(finishers["driverId"]))
].copy()

race_pits = pit_stops[
    (pit_stops["raceId"] == RACE_ID) &
    (pit_stops["driverId"].isin(finishers["driverId"]))
].copy()

pit_counts = race_pits.groupby("driverId").size()
finishers["estimated_stints"] = finishers["driverId"].map(pit_counts).fillna(0) + 1

lap_counts = race_laps.groupby("driverId").size()
finishers["lap_count"] = finishers["driverId"].map(lap_counts).fillna(0)

usable = finishers[finishers["estimated_stints"] >= 2].copy()
selected_drivers = (
    usable.sort_values(["estimated_stints", "lap_count"], ascending=False)
    .head(8)["driverId"].tolist()
)

if len(selected_drivers) < 5:
    raise ValueError("Fewer than 5 finishers have multiple stints. Choose another verified dry race.")

race_laps = race_laps[race_laps["driverId"].isin(selected_drivers)].copy()
race_pits = race_pits[race_pits["driverId"].isin(selected_drivers)].copy()
finishers_this_race = finishers[finishers["driverId"].isin(selected_drivers)].copy()

print(f"Selected {len(selected_drivers)} drivers: {selected_drivers}")
print(f"Recorded laps: {len(race_laps)}")
print(f"Pit stops: {len(race_pits)}")


## Step 2 — Calculate tire age
Tire age is the number of laps completed since the most recent pit stop. It resets after a pit stop. A `stint` number is also created.


In [ ]:
def add_tire_age(laps_df, pits_df):
    laps_df = laps_df.sort_values(["driverId", "lap"]).reset_index(drop=True)
    output = []

    for driver_id, group in laps_df.groupby("driverId"):
        group = group.sort_values("lap").copy()
        pit_laps = sorted(pits_df.loc[pits_df["driverId"] == driver_id, "lap"].tolist())
        last_pit_lap = 0
        pit_index = 0
        stint = 1

        for _, row in group.iterrows():
            lap_num = row["lap"]

            while pit_index < len(pit_laps) and lap_num > pit_laps[pit_index]:
                last_pit_lap = pit_laps[pit_index]
                pit_index += 1
                stint += 1

            row["tire_age"] = max(0, lap_num - last_pit_lap - 1)
            row["stint"] = stint
            output.append(row)

    return pd.DataFrame(output).reset_index(drop=True)

race_laps = add_tire_age(race_laps, race_pits)
print(race_laps[["driverId", "lap", "tire_age", "stint"]].head(15))


## Step 3 — Clean the lap data
Remove pit-stop laps, immediate following laps, and laps slower than 1.5× the driver's median lap time. The notebook reports every removal count.


In [ ]:
def clean_laps(laps_df, pits_df, multiplier=1.5):
    df = laps_df.copy()
    remove = set()

    for driver_id, group in pits_df.groupby("driverId"):
        for pit_lap in group["lap"]:
            remove.add((driver_id, pit_lap))
            remove.add((driver_id, pit_lap + 1))

    pit_mask = df.apply(lambda r: (r["driverId"], r["lap"]) in remove, axis=1)
    removed_pit = int(pit_mask.sum())
    df = df.loc[~pit_mask].copy()

    outlier_mask = pd.Series(False, index=df.index)
    for driver_id, group in df.groupby("driverId"):
        median = group["milliseconds"].median()
        outlier_mask.loc[group.index] = group["milliseconds"] > multiplier * median

    removed_outliers = int(outlier_mask.sum())
    clean = df.loc[~outlier_mask].copy()

    print("Laps before cleaning:", len(laps_df))
    print("Removed pit + out-laps:", removed_pit)
    print("Removed slow outliers:", removed_outliers)
    print("Laps remaining:", len(clean))
    return clean

clean_laps_df = clean_laps(race_laps, race_pits)

grid_lookup = finishers_this_race.set_index("driverId")["grid"]
clean_laps_df["grid"] = clean_laps_df["driverId"].map(grid_lookup)
clean_laps_df = clean_laps_df.dropna(
    subset=["milliseconds", "grid", "lap", "tire_age", "stint"]
).copy()


## Step 4 — Stint-based train/test split
For every driver, earlier stints are training data and the final stint is test data. We do not randomly shuffle laps because that could leak very similar conditions into both sets.


In [ ]:
train_parts = []
test_parts = []

for driver_id, group in clean_laps_df.groupby("driverId"):
    stints = sorted(group["stint"].unique())
    if len(stints) >= 2:
        final_stint = stints[-1]
        train_part = group[group["stint"] < final_stint]
        test_part = group[group["stint"] == final_stint]
        if len(train_part) > 0 and len(test_part) > 0:
            train_parts.append(train_part)
            test_parts.append(test_part)

if not train_parts:
    raise ValueError("No training data was produced. The selected race does not have usable multiple stints.")

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("Training laps:", len(train_df))
print("Testing laps:", len(test_df))


## Step 5 — Prepare features
Baseline = `grid + lap`
Enhanced = `grid + lap + tire_age`
Target = lap time in milliseconds.


In [ ]:
baseline_features = ["grid", "lap"]
enhanced_features = ["grid", "lap", "tire_age"]

X_train_baseline = train_df[baseline_features]
X_test_baseline = test_df[baseline_features]
X_train_enhanced = train_df[enhanced_features]
X_test_enhanced = test_df[enhanced_features]

y_train = train_df["milliseconds"]
y_test = test_df["milliseconds"]
results_list = []


## Step 6 — Train and evaluate four combinations
We compare Linear Regression and Random Forest using RMSE and MAE. Lower is better.


In [ ]:
def evaluate(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    mae = mean_absolute_error(y_test, pred)

    feature_set, model_name = name.split(" + ", 1)
    results_list.append({
        "Feature set": feature_set,
        "Model": model_name,
        "RMSE (ms)": round(rmse, 1),
        "MAE (ms)": round(mae, 1)
    })
    print(f"{name}: RMSE={rmse:.1f} ms, MAE={mae:.1f} ms")
    return model, pred

model_1, predictions_1 = evaluate(
    "Baseline + LinearRegression", LinearRegression(),
    X_train_baseline, X_test_baseline)

model_2, predictions_2 = evaluate(
    "Baseline + RandomForest",
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    X_train_baseline, X_test_baseline)

model_3, predictions_3 = evaluate(
    "Enhanced + LinearRegression", LinearRegression(),
    X_train_enhanced, X_test_enhanced)

model_4, predictions_4 = evaluate(
    "Enhanced + RandomForest",
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    X_train_enhanced, X_test_enhanced)


In [ ]:
comparison_table = pd.DataFrame(results_list).sort_values("RMSE (ms)").reset_index(drop=True)
display(comparison_table)
os.makedirs("outputs", exist_ok=True)
comparison_table.to_csv("outputs/model_comparison.csv", index=False)


## Step 7 — Effect of tire age
We compare each algorithm's baseline RMSE with its enhanced RMSE. This is calculated from the actual results.


In [ ]:
effect_rows = []

for algorithm in ["LinearRegression", "RandomForest"]:
    base = comparison_table[
        (comparison_table["Feature set"] == "Baseline") &
        (comparison_table["Model"] == algorithm)
    ]["RMSE (ms)"].iloc[0]

    enhanced = comparison_table[
        (comparison_table["Feature set"] == "Enhanced") &
        (comparison_table["Model"] == algorithm)
    ]["RMSE (ms)"].iloc[0]

    effect_rows.append({
        "Algorithm": algorithm,
        "Baseline RMSE (ms)": base,
        "Enhanced RMSE (ms)": enhanced,
        "RMSE change (%)": round((enhanced - base) / base * 100, 2)
    })

effect_table = pd.DataFrame(effect_rows)
display(effect_table)
effect_table.to_csv("outputs/tire_age_effect.csv", index=False)


## Step 8 — Final-stint visualization
Plot actual lap times against predictions for one driver's complete final stint.


In [ ]:
focus_driver_id = test_df["driverId"].iloc[0]
focus = test_df[test_df["driverId"] == focus_driver_id].sort_values("lap")

baseline_pred = model_2.predict(focus[baseline_features])
enhanced_pred = model_4.predict(focus[enhanced_features])

plt.figure(figsize=(9, 5))
plt.plot(focus["lap"], focus["milliseconds"], "o-", label="Actual lap time")
plt.plot(focus["lap"], baseline_pred, "--", label="Baseline prediction")
plt.plot(focus["lap"], enhanced_pred, "--", label="Enhanced (tire age) prediction")
plt.xlabel("Lap number")
plt.ylabel("Lap time (milliseconds)")
plt.title(f"Driver {focus_driver_id} — Final Stint: Predicted vs Actual")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/stint_prediction_plot.png", dpi=150, bbox_inches="tight")
plt.show()


## Step 9 — Conclusion
The conclusion below is based on the measured results, rather than assuming that tire age must improve prediction.


In [ ]:
best = comparison_table.iloc[0]
print("CONCLUSION")
print("===========")
print(f"Best model: {best['Feature set']} + {best['Model']}")
print(f"RMSE: {best['RMSE (ms)']:.1f} ms")
print(f"MAE: {best['MAE (ms)']:.1f} ms")

for _, row in effect_table.iterrows():
    direction = "decreased" if row["RMSE change (%)"] < 0 else "increased"
    print(f"{row['Algorithm']}: adding tire age {direction} RMSE by {abs(row['RMSE change (%)']):.2f}%.")
print("\nThese results apply to this selected race and driver sample; they should not be treated as a universal F1 rule.")
